<a href="https://colab.research.google.com/github/chanoosy/Test_task/blob/main/TestTask.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Библиотеки

In [31]:
import torch
import torchvision
import torchvision.transforms as transforms

import torch.nn as nn
from torchvision import models

from torch.utils.data import DataLoader, Subset
import torch.optim as optim


Фиксация Random Seed

In [32]:
import random

random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

Логирование

In [33]:
import datetime

def log(step, accuracy, seed=42):
    now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log_string = f"[{now}] Seed: {seed} | Step {step} | Accuracy: {accuracy:.4f}\n"

    with open("log.txt", "a") as f:
        f.write(log_string)

Получение датасета

In [34]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Resize((224, 224))])

train_set = torchvision.datasets.MNIST( root='./mnist', train=True, download=True, transform=transform)

test_set = torchvision.datasets.MNIST(root='./mnist', train=False, download=True, transform=transform)

Деление на ID(4 - 9) и OOD(0 - 3)

In [35]:
id = [4, 5, 6, 7, 8, 9]
ood = [0, 1, 2, 3]

id_train = []
ood_train = []

for i, label in enumerate(train_set.targets):
  if label in id:
    id_train.append(i)
  elif label in ood:
    ood_train.append(i)

id_test = []
ood_test = []

for i, label in enumerate(test_set.targets):
    if label in id:
        id_test.append(i)
    elif label in ood:
        ood_test.append(i)


Работа с ResNet-18: вызов ResNet-18, заморозка внутренних слов, дообучение на ID части

In [36]:
model = models.resnet18(weights='ResNet18_Weights.IMAGENET1K_V1')

for p in model.parameters():
    p.requires_grad = False

model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

nm_f = model.fc.in_features
model.fc = nn.Linear(nm_f, 6)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = torch.optim.SGD(filter(lambda p: p.requires_grad, model.parameters()), lr=0.01)
c_loss = nn.CrossEntropyLoss()

train_load = DataLoader(Subset(train_set, id_train), batch_size=64, shuffle=True)

epochs = 5

for ep in range(epochs):
    model.train()
    loss = 0.0
    correct = 0
    total = 0

    for img, lab in train_load:
        lab = lab - 4
        img, lab = img.to(device), lab.to(device)

        optimizer.zero_grad()
        outputs = model(img)
        batch_loss = c_loss(outputs, lab)
        batch_loss.backward()
        optimizer.step()

        loss += batch_loss.item()

        pred = outputs.argmax(dim=1)
        total += lab.size(0)
        correct += (pred == lab).sum().item()

    accuracy = 100 * correct / total
    avg_loss = loss / len(train_load)
    print(f"Epoch {ep + 1}/{epochs} | Accuracy: {accuracy:.2f}% | Loss: {avg_loss:.4f}")

Epoch 1/5 | Accuracy: 91.34% | Loss: 0.3843
Epoch 2/5 | Accuracy: 96.35% | Loss: 0.1542
Epoch 3/5 | Accuracy: 96.85% | Loss: 0.1202
Epoch 4/5 | Accuracy: 97.22% | Loss: 0.1055
Epoch 5/5 | Accuracy: 97.40% | Loss: 0.0944


Измерение точности на тутовой выборке ID

In [37]:
test_load = DataLoader(Subset(test_set, id_test), batch_size = 64, shuffle = False)

model.eval()

correct = 0
all = 0

with torch.no_grad():
  for img, lab in test_load:
    lab = lab - 4
    img = img.to(device)
    lab = lab.to(device)

    outputs = model(img)
    predskaz = outputs.argmax(dim = 1)

    all += len(lab)
    correct += (predskaz == lab).sum().item()

accuracy = 100 * correct / all
print(f"Accuracy on ID test: {accuracy:.4f}%")
log(5, accuracy)


Accuracy on ID test: 96.7825%


Функция рассчета энергии

In [38]:
def energy(logits, T = 1):
  energy = -T * torch.logsumexp(logits / T, dim = 1)
  return energy

Нахождение порога (сравнением энергий для ID и OOD)

In [39]:
id_load = DataLoader(Subset(train_set, id_train), batch_size = 64, shuffle = False)
ood_load = DataLoader(Subset(train_set, ood_train), batch_size=64, shuffle=False)

model.eval()
id_energy = []
ood_energy = []

with torch.no_grad():
  for img, label in id_load:
    img = img.to(device)
    outputs = model(img)
    e = energy(outputs)
    id_energy += e.tolist()

with torch.no_grad():
  for img, label in ood_load:
    img = img.to(device)
    outputs = model(img)
    e = energy(outputs)
    ood_energy += e.tolist()




In [40]:
all_energy = sorted(id_energy + ood_energy)

best_t = 0
min_error = len(id_energy) + len(ood_energy)

for i in range (0, len(all_energy), 10):
  t = all_energy[i]

  false_ood = sum(1 for e in id_energy if e > t)
  false_id = sum(1 for e in ood_energy if e <= t)

  all_errors = false_ood + false_id

  if all_errors < min_error:
    min_error = all_errors
    best_t = t
print(f"Minimal number of mistakes: {min_error} / {len(all_energy)}")
print(f"Optimal threshold: {best_t:.4f}")

Minimal number of mistakes: 11009 / 60000
Optimal threshold: -4.6313


Energy-based OOD детектор

In [41]:
def detector (model, image, threshold, T = 1):
  model.eval()

  with torch.no_grad():
    logits = model(image)
    e = energy(logits, T)

    is_id = (e <= threshold).int()
  return is_id, e

Оценка точности на подвыборке
состоящих из тестовых частей OOD и ID примеров исходной в соотношении 1 к
1 из не менее чем 1000 изображений в сумме

In [42]:
id_part = id_test[1000:1500]
ood_part = ood_test[1000:1500]

all = id_part + ood_part
all_load = DataLoader(Subset(test_set, all), batch_size = 64, shuffle = False)

correct = 0
total = 0

model.eval()

for img, lab in all_load:
  img = img.to(device)

  is_id, e = detector(model, img, best_t)
  right_lab = (lab >= 4).int().to(device)
  matched = (is_id == right_lab).sum().item()

  correct += matched
  total += len(img)

accuracy = 100 * correct / total
print(f"For {total} images the final accuracy is: {accuracy:.4f}%")
log(7, accuracy)

For 1000 images the final accuracy is: 80.4000%


FGSM атака

In [43]:
def attack_detector(image, epsilon, grad):
  direction = grad.sign()
  fake_image = image - epsilon * direction
  fake_image = torch.clamp(fake_image, 0, 1)

  return fake_image


In [44]:
epsilon = 8/255
wrong = 0
all = 0

model.eval()

ood_loader = DataLoader(Subset(test_set, ood_part), batch_size = 64, shuffle = False)

for img, lab in ood_loader:
  img = img.to(device)
  img.requires_grad = True

  logits = model(img)
  curr_energy = energy(model(img))

  model.zero_grad()
  curr_energy.sum().backward()

  fake_img = img - epsilon * img.grad.data.sign()
  fake_img = torch.clamp(fake_img, 0, 1)

  with torch.no_grad():
    new_energy = energy(model(fake_img))
    wrong += (new_energy <= best_t).sum().item()
    all+=len(img)

print(f"Fooled: {wrong} / 600")

Fooled: 485 / 600


Повторная оценка точности с OOD частью, подверженной атаке

In [45]:
model.eval()
epsilon = 8/255
correct = 0
all = 0

id_loader = DataLoader(Subset(test_set, id_part), batch_size = 64)
for img, lab in id_loader:
  img = img.to(device)
  with torch.no_grad():
    en = energy(model(img))
    correct += (en <= best_t).sum().item()
    all += len(img)

ood_loader = DataLoader(Subset(test_set, ood_part), batch_size = 64)
for img, lab in ood_loader:
  img = img.to(device)
  img.requires_grad = True

  curr_en = energy(model(img))
  model.zero_grad()
  curr_en.sum().backward()

  fake_img = img - epsilon * img.grad.data.sign()
  fake_img = torch.clamp(fake_img, 0, 1)

  with torch.no_grad():
    new_en = energy(model(fake_img))
    correct += (new_en > best_t).sum().item()
    all += len(img)

accuracy = 100 * correct / all
print(f"Accuracy under attack: {accuracy:.4f}%")
log(10, accuracy)

Accuracy under attack: 42.0000%
